In [2]:
!pip install torch torchvision timm

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

In [ ]:
train_csv = "/kaggle/input/datasets/afreen/aicliniscan/train.csv"
img_dir = "/kaggle/input/datasets/afreen/aicliniscan/train"
df = pd.read_csv(train_csv)

# Remove "No Abnormalities"

In [5]:
df = df[df["class_id"] != 14]

In [6]:
# Create Image  Single Label Mapping
df = df.sort_values("image_id")

df_single = df.groupby("image_id").first().reset_index()

print("Total images:", len(df_single))

Total images: 4394


In [7]:
class XrayDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        img_path = os.path.join(self.img_dir, row["image_id"] + ".png")
        image = Image.open(img_path).convert("RGB")
        
        label = int(row["class_id"])
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [8]:
#Transforms
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

# Create Dataset + Loader

In [9]:
dataset = XrayDataset(df_single, img_dir, transform)

train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

# Classification Model (ResNet50)

In [10]:
model = models.resnet50(pretrained=True)

model.fc = nn.Linear(model.fc.in_features, 14)

model = model.cuda()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 214MB/s]


In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [12]:
epochs = 5

for epoch in range(epochs):
    
    model.train()
    total_loss = 0
    
    for images, labels in train_loader:
        
        images = images.cuda()
        labels = labels.cuda()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 562.3804
Epoch 2, Loss: 495.7861
Epoch 3, Loss: 427.4897
Epoch 4, Loss: 293.0826
Epoch 5, Loss: 134.9698


In [13]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import torch.nn.functional as F

In [14]:
model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    
    for images, labels in train_loader:
        
        images = images.cuda()
        labels = labels.cuda()
        
        outputs = model(images)
        
        probs = F.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

In [15]:
accuracy = accuracy_score(all_labels, all_preds)
print("Accuracy:", accuracy)

Accuracy: 0.9301319981793355


In [16]:
f1 = f1_score(all_labels, all_preds, average='macro')
print("F1 Score:", f1)

F1 Score: 0.8346002606567324


In [17]:
auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
print("AUC:", auc)

AUC: 0.9976688402900283


In [18]:
torch.save(model.state_dict(), "/kaggle/working/cliniscan_classification_model.pth")

print("Model saved successfully")

Model saved successfully


In [19]:
import os

print(os.listdir("/kaggle/working"))

['cliniscan_classification_model.pth', '__notebook__.ipynb']


In [20]:
from IPython.display import FileLink

FileLink("/kaggle/working/cliniscan_classification_model.pth")

/kaggle/working/cliniscan_classification_model.pth

In [21]:
class_names = {
 0: "Aortic enlargement",
 1: "Atelectasis",
 2: "Calcification",
 3: "Cardiomegaly",
 4: "Consolidation",
 5: "ILD",
 6: "Infiltration",
 7: "Lung Opacity",
 8: "Nodule/Mass",
 9: "Other lesion",
 10: "Pleural effusion",
 11: "Pleural thickening",
 12: "Pneumothorax",
 13: "Pulmonary fibrosis"
}

import json

with open("/kaggle/working/class_names.json", "w") as f:
    json.dump(class_names, f)

In [22]:
FileLink("/kaggle/working/class_names.json")

/kaggle/working/class_names.json